In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df=pd.read_pickle(f"./df.pkl")
y=df[['dm']]
X=df.drop(columns=['dm'])

In [19]:
X.shape

(13796, 25)

In [20]:
X_trainvalid, X_test, y_trainvalid, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )

In [21]:
X_trainvalid.shape

(11036, 25)

In [22]:
X_train,X_valid, y_train, y_valid=train_test_split(X_trainvalid, y_trainvalid, test_size=0.2, random_state= 65, shuffle=True)

In [23]:
X_valid.shape, y_valid.shape

((2208, 25), (2208, 1))

In [24]:
from model.optimize_state import safety_adjust_delta
def apply_safety_adjustment(x, raw_deltas):
    patient = x.iloc[0].to_dict()
    adjusted_deltas = {}

    for var, delta in raw_deltas.items():
        result = safety_adjust_delta(patient, var, float(delta))
        adjusted_delta = float(result["adjusted_delta"])
        adjusted_deltas[var] = adjusted_delta

    return adjusted_deltas

In [25]:
from model.progression_scoring import progression_scoring
from model.optimize_state import compute_total_decision

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

epsilons = [3,5,7,9]
lambdas = [0.001,0.005,0.01,0.05,0.1,0.2,0.5,1.0]

results=[]

X_eval = X_valid.sample(100, random_state=42)

BASE_DIR = Path.cwd()

model_paths = [
            str(BASE_DIR / "model" / "final_checkpoints" / f"fold_{i}" / "best-checkpoint-v6.ckpt")
            for i in range(5)
        ]

with open(BASE_DIR / "scalers.pkl", "rb") as f:
    scalers = pickle.load(f)
    
def apply_deltas(X, deltas):
        X_opt = X.copy()

        for col, delta in deltas.items():
            if col in X_opt.columns:
                X_opt[col] = X_opt[col] + delta

        return X_opt


for eps, lam in itertools.product(epsilons,lambdas):

    score_reduction=[]
    l1_changes=[]
    n_changes=[]
    ratio_changes = []
    score_increased = []
    walk_max = []
    empty_recommendation = []

    for idx in range(len(X_eval)):

        x = X_eval.iloc[[idx]]

        before = progression_scoring(
            x,
            model_paths,
            scalers
        )

        raw_deltas = compute_total_decision(
            x,
            model_paths,
            scalers,
            epsilon=eps,
            lambda_reg=lam
        )
        
        deltas=apply_safety_adjustment(x,raw_deltas)

        x_after = apply_deltas(x,deltas)

        after = progression_scoring(
            x_after,
            model_paths,
            scalers
        )
        
        inv = {
        'wk_smk': (0.0, 420.0),
        'wk_alc': (0.0, 40.0),
        'wk_mvpa_play': (0.0, 300.0),
        'wk_walk': (0.0, 1260.0),
        'wk_sleep': (360.0, 540.0),
        'stress': (1.0, 4.0),
        'wk_break': (0.0, 6.0),
        'wk_lunch': (0.0, 6.0),
        'wk_dinner': (0.0, 6.0),
        'wk_veg1': (0.0, 21.0),
        'wk_veg2': (0.0, 21.0),
        'wk_fruit': (0.0, 21.0),
        }

        ratios=[]
        for var, delta in deltas.items():

            delta = float(delta)
            min_val, max_val = inv[var]
            scale = max_val - min_val

            ratio = abs(delta) / scale
            ratios.append(ratio)

        score_reduction.append(before-after)
        
        score_increased.append(after > before)

        l1_changes.append(
                np.sum(np.abs(list(deltas.values())))
            )

        n_changes.append(len(deltas))

        ratio_changes.append(np.mean(ratios))
        
        empty_recommendation.append(len(deltas) == 0)

        # walk가 상한(1260)에 도달했는지
        walk_max.append(
            x_after.iloc[0]["wk_walk"] >= 420 - 1e-6
        )

    results.append({

        "epsilon":eps,
        "lambda":lam,

        "score_reduction":
        np.mean(score_reduction),

        "L1_change":
        np.mean(l1_changes),

        "n_modified":
        np.mean(n_changes),

        "mean_ratio_change": np.mean(ratio_changes),
        
        "score_increase_rate": np.mean(score_increased),
        
        "walk_max_rate": np.mean(walk_max),

        "empty_recommendation_rate": np.mean(empty_recommendation)

    })

results=pd.DataFrame(results)

c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning

In [26]:
results

,epsilon,lambda,score_reduction,L1_change,n_modified,mean_ratio_change,score_increase_rate,walk_max_rate,empty_recommendation_rate
0,3,0.001,1.646936,323.825616,6.81,0.173914,0.01,0.48,0.0
1,3,0.005,1.646121,322.326182,6.78,0.173893,0.01,0.48,0.0
2,3,0.010,1.646703,315.651167,6.75,0.171599,0.01,0.48,0.0
3,3,0.050,1.641260,305.064125,6.81,0.165627,0.02,0.48,0.0
4,3,0.100,1.650964,286.599190,6.73,0.160871,0.00,0.45,0.0
5,3,0.200,1.648164,244.299638,6.57,0.153616,0.01,0.43,0.0
6,3,0.500,1.560410,158.620908,6.19,0.128271,0.01,0.36,0.0
7,3,1.000,1.379297,53.796868,5.60,0.103381,0.10,0.23,0.0
8,5,0.001,1.695820,370.105058,6.87,0.196203,0.01,0.45,0.0
9,5,0.005,1.695518,368.786341,6.87,0.195788,0.01,0.45,0.0


In [ ]:
import inspect
import model.optimize_state as optimize_state

print(optimize_state.__file__)
print(optimize_state.get_feature_deltas)
print(inspect.getsource(optimize_state.get_feature_deltas))

c:\Users\cmc\Desktop\T2D_AI_research\model\optimize_state.py
<function get_feature_deltas at 0x0000019B0FACBB00>
def optimize_with_mlp(
    model, scaler, x0_np, columns, device="cuda",
    lr=0.01, steps=300, lambda_reg=0.01, epsilon=5,
    fixed_features=['sex','age','edu','income','job','glu','hba1c','sbp','bmi','hdl','tg','ldl','wc'],
    clamp_dict = {
    'wk_smk': (0.0, 420.0),
    'wk_alc': (0.0, 40.0),
    'wk_mvpa_play': (0.0, 300.0), #mvpa는 최대 300분
    'wk_walk': (0.0, 1260.0),
    'wk_sleep': (360.0, 540.0),  #constraint
    'stress': (1.0, 4.0),
    'wk_break': (0.0, 6.0),
    'wk_lunch': (0.0, 6.0),
    'wk_dinner': (0.0, 6.0),
    'wk_veg1': (0.0, 21.0),
    'wk_veg2': (0.0, 21.0),
    'wk_fruit': (0.0, 21.0),
    }
):
    direction_constraints = {
    "wk_walk": "increase",
    "wk_mvpa_play":"increase",
    "stress": "decrease",
    "wk_alc": "decrease",
    "wk_smk": "decrease",
    "wk_veg1": "increase",
    "wk_veg2": "increase",
    }

    model.eval()
    x0_np = 

In [29]:
import importlib
import model.optimize_state as optimize_state

optimize_state = importlib.reload(optimize_state)

print(optimize_state.get_feature_deltas.__name__)
print(inspect.getsource(optimize_state.get_feature_deltas))

get_feature_deltas
def get_feature_deltas(
    x0,
    x_opt,
    columns,
    fixed_features=['sex','age','edu','income','job','glu','hba1c','sbp','bmi','hdl','tg','ldl','wc'],
    tol=1e-6,
    min_abs_ratio=0.01
    ):
    
    clamp_dict = {
    'wk_smk': (0.0, 420.0),
    'wk_alc': (0.0, 40.0),
    'wk_mvpa_play': (0.0, 300.0), #mvpa는 최대 300분
    'wk_walk': (0.0, 1260.0),
    'wk_sleep': (360.0, 540.0),  #constraint
    'stress': (1.0, 4.0),
    'wk_break': (0.0, 6.0),
    'wk_lunch': (0.0, 6.0),
    'wk_dinner': (0.0, 6.0),
    'wk_veg1': (0.0, 21.0),
    'wk_veg2': (0.0, 21.0),
    'wk_fruit': (0.0, 21.0),
    }

    x0 = np.array(x0).reshape(-1)
    x_opt = np.array(x_opt).reshape(-1)

    deltas = {}

    for i, col in enumerate(columns):
        if col in fixed_features:
            continue

        diff = x_opt[i] - x0[i]

        if abs(diff) <= tol:
            continue
        
        lo, hi = clamp_dict[col]
        scale = hi - lo

        normalized_change = abs(diff